In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis
import numpy as np
import os
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings

warnings.filterwarnings('ignore')

csv_file = 'analitic_avito_dataset_50_pages.csv'

df = pd.read_csv(csv_file)

property_type_col = 'type'
space_col = 'space'
price_col = 'price'
area_price_col = 'area_price'
time_col = 'time_to_metro'
date_col = 'date'
link_col = 'link'
metro_col = 'metro'

def clean_currency_col(col):
    if col in df.columns and pd.api.types.is_string_dtype(df[col]):
        df[col] = df[col].str.replace(r'[^0-9]', '', regex=True).astype(float)

clean_currency_col(price_col)
clean_currency_col(area_price_col)

if date_col in df.columns:
    df[date_col] = pd.to_datetime(df[date_col], format='%d.%m.%Y', errors='coerce')

if price_col in df.columns:
    Q1 = df[price_col].quantile(0.25)
    Q3 = df[price_col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = max(0, Q1 - 1.5 * IQR)
    upper_bound = Q3 + 1.5 * IQR
    df = df[(df[price_col] >= lower_bound) & (df[price_col] <= upper_bound)]

numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

if time_col in df.columns:
    time_min, time_max = df[time_col].min(), df[time_col].max()
    num_bins = 5
    bins = np.linspace(time_min, time_max, num_bins + 1)
    labels = [f'{int(bins[i])}-{int(bins[i+1])}' for i in range(len(bins)-1)]
    df['time_bin'] = pd.cut(df[time_col], bins=bins, labels=labels, include_lowest=True)

output_dir = 'analysis_plots'
os.makedirs(output_dir, exist_ok=True)

In [18]:
if price_col in df.columns and space_col in df.columns and area_price_col in df.columns and time_col in df.columns:
    num_vars = [price_col, space_col, area_price_col, time_col]
    table = pd.DataFrame({
    'Переменная': num_vars,
    'Среднее': df[num_vars].mean().round(2),
    })

print(table.to_string(index=False))

   Переменная     Среднее
        price 10839052.33
        space       45.49
   area_price   243335.46
time_to_metro       19.00


In [19]:
if price_col in df.columns and space_col in df.columns and area_price_col in df.columns and time_col in df.columns and metro_col in df.columns:
    num_vars = [price_col, space_col, area_price_col, time_col]
    group_means=df.groupby(metro_col)[num_vars].mean().round(2)
    group_means['количество_объявлений'] = df.groupby(metro_col).size()
    group_means = group_means.sort_values(by=space_col, ascending=False)
    
    print(group_means.to_string())

                               price  space  area_price  time_to_metro  количество_объявлений
metro                                                                                        
Елизаровская             16536000.00  74.60   235398.80          13.00                      5
Новочеркасская           14918120.17  63.38   238463.96          19.62                     24
Садовая                  14729411.76  63.24   232070.41          11.18                     17
Гостиный двор            18700000.00  63.00   296825.00          15.00                      1
Технологический ин-т I   13705333.33  62.33   238875.67          10.00                      3
Чернышевская             15729826.09  59.43   281246.04          13.70                     23
Академическая            11996875.00  55.42   218588.31          22.10                     48
Удельная                 12679596.00  53.32   241971.28          19.60                     25
Ладожская                11050685.13  53.31   214517.80     

In [20]:
mu0 = 15
t_stat, p_val = stats.ttest_1samp(df['time_to_metro'].dropna(), popmean=mu0)

print(f"t-статистика: {t_stat:.4f}")
print(f"p-value: {p_val:.4f}")
print(f"Среднее в выборке: {df['time_to_metro'].mean():.2f}")

if p_val < 0.05:
    print("Отвергаем H₀: среднее значимо отличается от 15 минут")
else:
    print("Нет оснований отвергать H₀")

t-статистика: 18.0879
p-value: 0.0000
Среднее в выборке: 19.00
Отвергаем H₀: среднее значимо отличается от 15 минут


In [21]:
m0 = 10_000_000
data = df['price'].dropna()

n_above = (data > m0).sum()
n_below = (data < m0).sum()
n_total = n_above + n_below

p_sign = 2 * stats.binom.cdf(min(n_above, n_below), n_total, 0.5)

print(f"Наблюдений выше m0: {n_above}")
print(f"Наблюдений ниже m0: {n_below}")
print(f"p-value (критерий знаков): {p_sign:.4f}")
print(f"Медиана в выборке: {data.median():,.0f}")

if p_sign < 0.05:
    print("Отвергаем H₀: медиана значимо отличается от 10 млн")
else:
    print("Нет оснований отвергать H₀")

Наблюдений выше m0: 878
Наблюдений ниже m0: 901
p-value (критерий знаков): 0.6020
Медиана в выборке: 9,990,000
Нет оснований отвергать H₀
